# NASA Space Apps Challenge — Pipeline Học máy Toàn cầu `MODIS -> VIIRS` (Bản CPU Đa nhân)
* **Tập trung 100% vào AI & Xuất dữ liệu Toàn cầu (`2000–2025`, H3 `res=7`, 18 cột chuẩn).**
* **Phần cứng:** Chạy hoàn toàn trên **CPU đa nhân** (`HuberRegressor`, `HistGradientBoosting`, `LightGBM`, `XGBoost CPU hist`, `Weighted CPU Ensemble + EQM`) — không cần GPU.
* **Chia tập:** `Train 2013–2023` (`660,000` cặp; lấy `2023` học trọng số NNLS & `2018–2023` khớp EQM) | `Test 2024–2025` (`200,000` cặp).

## Ô 1: Cấu hình Đường dẫn Toàn cầu & Trích xuất Cặp Trùng (2013–2023 | 2024–2025)

In [1]:
import importlib.util
import subprocess
import sys

for mod_name, pip_name in [("duckdb", "duckdb"), ("lightgbm", "lightgbm"), ("xgboost", "xgboost"), ("sklearn", "scikit-learn"), ("scipy", "scipy"), ("joblib", "joblib")]:
    if importlib.util.find_spec(mod_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import os
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
import duckdb
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from scipy.optimize import nnls
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG") or os.path.isdir("/content"))
N_CPU_JOBS = os.cpu_count() or 4
print(f"CPU Active: cores={N_CPU_JOBS} | Colab={IS_COLAB} | LightGBM={lgb.__version__} | XGBoost={xgb.__version__}")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BASE = "/content/drive/MyDrive/NasaHackathonData"
    H3_WORLD_ROOT = f"{DRIVE_BASE}/parquet_h3_world"
    HARMONIZED_WORLD_ROOT = f"{DRIVE_BASE}/parquet_h3_world_harmonized"
    OUTPUT_DIR = f"{DRIVE_BASE}/outputs"
else:
    H3_WORLD_ROOT = "D:/NasaHackathonData/parquet_h3_world"
    HARMONIZED_WORLD_ROOT = "D:/NasaHackathonData/parquet_h3_world_harmonized"
    OUTPUT_DIR = "c:/Uni/261/Space Apps Challenge/outputs"

if not (os.path.isdir(f"{H3_WORLD_ROOT}/modis") and os.path.isdir(f"{H3_WORLD_ROOT}/viirs")):
    raise FileNotFoundError(f"Không tìm thấy dữ liệu H3 toàn cầu tại: {H3_WORLD_ROOT}/modis và /viirs")

os.makedirs(HARMONIZED_WORLD_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_OUT_PATH = f"{OUTPUT_DIR}/ml_cpu_modis_to_viirs_bundle.joblib"
COMPARISON_OUT_CSV = f"{OUTPUT_DIR}/ml_cpu_model_comparison.csv"


def sample_global_pairs(years: list[int], per_year: int, seed_base: int) -> pd.DataFrame:
    con = duckdb.connect()
    con.execute("SET memory_limit='6GB';")
    try:
        frames = []
        for yr in years:
            frames.append(con.execute(f"""
                WITH m AS (
                    SELECT h3_res7, acq_date, year, daynight, country, center_lat,
                           type, bright_mir, bright_tir, GREATEST(frp, 0.0) AS frp,
                           point_count, conf_l, conf_n, conf_h
                    FROM read_parquet('{H3_WORLD_ROOT}/modis/{yr}.parquet')
                    WHERE bright_mir BETWEEN 200.0 AND 550.0 AND bright_tir BETWEEN 180.0 AND 430.0 AND frp >= 0.0
                ),
                v AS (
                    SELECT h3_res7, acq_date, daynight, country,
                           bright_mir AS v_mir, bright_tir AS v_tir, GREATEST(frp, 0.0) AS v_frp
                    FROM read_parquet('{H3_WORLD_ROOT}/viirs/{yr}.parquet')
                    WHERE bright_mir BETWEEN 200.0 AND 367.0 AND bright_tir BETWEEN 180.0 AND 430.0 AND frp >= 0.0
                )
                SELECT m.*, v.v_mir, v.v_tir, v.v_frp
                FROM m JOIN v USING (h3_res7, acq_date, daynight, country)
                USING SAMPLE {per_year} ROWS (reservoir, {seed_base + yr})
            """).df())
        return pd.concat(frames, ignore_index=True)
    finally:
        con.close()


t0 = time.time()
df_train = sample_global_pairs(list(range(2013, 2024)), per_year=60_000, seed_base=100)
df_test = sample_global_pairs(list(range(2024, 2026)), per_year=100_000, seed_base=900)
print(f"Sampled Global Train (2013-2023): {len(df_train):,} | Test (2024-2025): {len(df_test):,} in {time.time() - t0:.1f}s")

CPU Active: cores=8 | Colab=True | LightGBM=4.6.0 | XGBoost=3.4.1
Mounted at /content/drive


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (5 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sampled Global Train (2013-2023): 660,000 | Test (2024-2025): 200,000 in 192.0s


## Ô 2: 19 Đặc trưng, Chuẩn hóa & Bộ Mô hình CPU (`Huber`, `HistGB`, `LightGBM`, `XGBoost CPU`, `EQM`)

In [2]:
TREE_NAMES = ["HistGradientBoosting", "LightGBM", "XGBoost CPU"]


def build_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    """Build 19 physics, per-point confidence ratio, type one-hot, latitude & cyclical month features."""
    pts = np.maximum(df["point_count"].to_numpy(dtype=np.float32), 1.0)
    mir = df["bright_mir"].to_numpy(dtype=np.float32)
    tir = df["bright_tir"].to_numpy(dtype=np.float32)
    frp = np.maximum(df["frp"].to_numpy(dtype=np.float32), 0.0)
    cl = df["conf_l"].to_numpy(dtype=np.float32)
    cn = df["conf_n"].to_numpy(dtype=np.float32)
    ch = df["conf_h"].to_numpy(dtype=np.float32)
    day = (df["daynight"].to_numpy() == "D").astype(np.float32)
    tp = df["type"].to_numpy(dtype=np.int32)
    abs_lat = np.abs(df["center_lat"].to_numpy(dtype=np.float32))
    m_rad = (2.0 * np.pi / 12.0) * pd.to_datetime(df["acq_date"]).dt.month.to_numpy(dtype=np.float32)

    return np.column_stack([
        mir, tir, mir - tir,
        np.log1p(frp), frp / pts,
        pts, cl, cn, ch, ch / pts, cl / pts,
        day,
        (tp == 0).astype(np.float32), (tp == 1).astype(np.float32),
        (tp == 2).astype(np.float32), (tp == 3).astype(np.float32),
        abs_lat, np.sin(m_rad), np.cos(m_rad),
    ])


def make_cpu_tree_suite(n_jobs: int = N_CPU_JOBS) -> dict:
    return {
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=250, max_depth=9, max_leaf_nodes=63, min_samples_leaf=50, learning_rate=0.07, random_state=42
        ),
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=300, num_leaves=63, learning_rate=0.07, subsample=0.85,
            subsample_freq=1, colsample_bytree=0.85, force_col_wise=True, n_jobs=min(n_jobs, 8), random_state=42, verbose=-1
        ),
        "XGBoost CPU": xgb.XGBRegressor(
            n_estimators=350, max_depth=8, max_bin=256, learning_rate=0.06,
            subsample=0.85, colsample_bytree=0.85, tree_method="hist", device="cpu", n_jobs=min(n_jobs, 8), random_state=42
        ),
    }


def optimize_weights(preds_mat: np.ndarray, y_val: np.ndarray) -> np.ndarray:
    w, _ = nnls(preds_mat, y_val)
    s = float(np.sum(w))
    return (w / s) if s > 1e-9 else np.full(preds_mat.shape[1], 1.0 / preds_mat.shape[1], dtype=np.float64)


def fit_eqm_map(pred_ref: np.ndarray, true_ref: np.ndarray, n_q: int = 2001) -> dict[str, np.ndarray]:
    probs = np.linspace(0.0, 1.0, n_q, dtype=np.float64)
    return {"probs": probs, "src_q": np.quantile(pred_ref, probs), "tgt_q": np.quantile(true_ref, probs)}


def apply_eqm_map(preds: np.ndarray, eqm_map: dict[str, np.ndarray]) -> np.ndarray:
    return np.interp(np.interp(preds, eqm_map["src_q"], eqm_map["probs"]), eqm_map["probs"], eqm_map["tgt_q"])

## Ô 3: Huấn luyện CPU Đa nhân (2013–2023), Đánh giá (2024–2025) & Lưu Bundle

In [3]:
X_train = build_feature_matrix(df_train)
X_test = build_feature_matrix(df_test)

train_yrs = df_train["year"].to_numpy()
val_mask = train_yrs == 2023
sub_mask = ~val_mask  # 2013-2022 -> predict 2023 for leak-free NNLS weights
ref_mask = train_yrs >= 2018  # Dual-VIIRS baseline (2018-2023)

TARGET_SPECS = [
    ("bright_mir", "v_mir", "bright_mir", False, 200.0, 367.0, True),
    ("bright_tir", "v_tir", "bright_tir", False, 180.0, 425.0, False),
    ("frp_log1p", "v_frp", "frp", True, 0.0, float(np.log1p(55000.0)), True),
]


def eval_row(target_name: str, model_name: str, y_true: np.ndarray, y_pred: np.ndarray, raw_modis: np.ndarray) -> dict:
    return {
        "target": target_name,
        "model": model_name,
        "mae": round(float(mean_absolute_error(y_true, y_pred)), 4),
        "rmse": round(float(root_mean_squared_error(y_true, y_pred)), 4),
        "r2": round(float(r2_score(y_true, y_pred)), 4),
        "bias_pct": round(float((np.sum(y_pred) - np.sum(y_true)) / max(np.sum(y_true), 1e-9) * 100.0), 2),
        "true_median": round(float(np.median(y_true)), 3),
        "pred_median": round(float(np.median(y_pred)), 3),
        "modis_median": round(float(np.median(raw_modis)), 3),
        "true_mean": round(float(np.mean(y_true)), 3),
        "pred_mean": round(float(np.mean(y_pred)), 3),
    }


comparison_rows = []
bundle = {"feature_count": X_train.shape[1], "tree_names": TREE_NAMES, "models": {}}

for tgt_key, v_col, m_col, is_log, lo, hi, use_eqm in TARGET_SPECS:
    t_s = time.time()
    y_tr_mw = np.clip(df_train[v_col].to_numpy(dtype=np.float64), 0.0 if is_log else lo, 55000.0 if is_log else hi)
    y_te_mw = np.clip(df_test[v_col].to_numpy(dtype=np.float64), 0.0 if is_log else lo, 55000.0 if is_log else hi)
    raw_te_mw = np.maximum(df_test[m_col].to_numpy(dtype=np.float64), 0.0)

    y_tr = np.log1p(y_tr_mw) if is_log else y_tr_mw
    y_te = np.log1p(y_te_mw) if is_log else y_te_mw
    raw_te = np.log1p(raw_te_mw) if is_log else raw_te_mw

    # Step A: Leak-free NNLS weights (2013-2022 -> 2023)
    sub_suite = make_cpu_tree_suite()
    val_cols = []
    for m_name in TREE_NAMES:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sub_suite[m_name].fit(X_train[sub_mask], y_tr[sub_mask])
            val_cols.append(np.clip(sub_suite[m_name].predict(X_train[val_mask]), lo, hi))
    weights = optimize_weights(np.column_stack(val_cols), y_tr[val_mask])

    # Step B: Fit full CPU models (2013-2023)
    huber = make_pipeline(StandardScaler(), HuberRegressor(epsilon=1.35, max_iter=300)).fit(X_train, y_tr)
    pred_map = {
        "Uncalibrated MODIS": raw_te,
        "HuberRegressor": np.clip(huber.predict(X_test), lo, hi),
    }

    full_suite = make_cpu_tree_suite()
    te_cols, ref_cols = [], []
    for m_name in TREE_NAMES:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_suite[m_name].fit(X_train, y_tr)
            p_te = np.clip(full_suite[m_name].predict(X_test), lo, hi)
            p_ref = np.clip(full_suite[m_name].predict(X_train[ref_mask]), lo, hi)
        pred_map[m_name] = p_te
        te_cols.append(p_te)
        ref_cols.append(p_ref)
        if use_eqm:
            pred_map[f"{m_name} + EQM"] = np.clip(apply_eqm_map(p_te, fit_eqm_map(p_ref, y_tr[ref_mask])), lo, hi)

    p_te_ens = np.clip(np.column_stack(te_cols) @ weights, lo, hi)
    pred_map["Weighted CPU Ensemble"] = p_te_ens

    eqm_map = None
    if use_eqm:
        p_ref_ens = np.clip(np.column_stack(ref_cols) @ weights, lo, hi)
        eqm_map = fit_eqm_map(p_ref_ens, y_tr[ref_mask])
        pred_map["CPU Ensemble + EQM"] = np.clip(apply_eqm_map(p_te_ens, eqm_map), lo, hi)

    for label, p_val in pred_map.items():
        comparison_rows.append(eval_row(tgt_key, label, y_te, p_val, raw_te))
    if is_log:
        for label, p_val in pred_map.items():
            comparison_rows.append(eval_row("frp_mw", label, y_te_mw, np.expm1(p_val), raw_te_mw))

    bundle["models"][tgt_key] = {
        "trees": full_suite, "huber": huber, "weights": weights,
        "is_log": is_log, "clip_lo": lo, "clip_hi": hi, "eqm_map": eqm_map,
    }
    print(f"[{tgt_key}] weights={dict(zip(TREE_NAMES, [round(float(x), 3) for x in weights]))} | time={time.time() - t_s:.1f}s")

df_cmp = pd.DataFrame(comparison_rows)
df_cmp.to_csv(COMPARISON_OUT_CSV, index=False)
joblib.dump(bundle, MODEL_OUT_PATH, compress=3)
print(f"\nSaved: {MODEL_OUT_PATH} ({os.path.getsize(MODEL_OUT_PATH) / 1e6:.2f} MB)")
print(df_cmp.to_string(index=False))

[bright_mir] weights={'HistGradientBoosting': 0.413, 'LightGBM': 0.292, 'XGBoost CPU': 0.296} | time=63.1s
[bright_tir] weights={'HistGradientBoosting': 0.372, 'LightGBM': 0.263, 'XGBoost CPU': 0.365} | time=69.5s
[frp_log1p] weights={'HistGradientBoosting': 0.362, 'LightGBM': 0.274, 'XGBoost CPU': 0.364} | time=66.1s

Saved: /content/drive/MyDrive/NasaHackathonData/outputs/ml_cpu_modis_to_viirs_bundle.joblib (10.20 MB)
    target                      model     mae     rmse      r2  bias_pct  true_median  pred_median  modis_median  true_mean  pred_mean
bright_mir         Uncalibrated MODIS 24.4024  28.2737 -2.3611     -5.91      347.910      323.400       323.400    347.721    327.160
bright_mir             HuberRegressor  9.8956  12.5165  0.3413     -0.56      347.910      346.635       323.400    347.721    345.777
bright_mir       HistGradientBoosting  9.6734  12.1488  0.3794     -0.44      347.910      347.512       323.400    347.721    346.192
bright_mir HistGradientBoosting + EQ

## Ô 4: Xuất Bộ Dữ liệu Harmonized Toàn cầu 26 Năm (`2000–2025`, 18 cột) bằng CPU Đa nhân

In [ ]:
SCHEMA_COLS_SQL = """
    CAST(acq_date AS DATE) AS acq_date, CAST(year AS SMALLINT) AS year,
    CAST(daynight AS VARCHAR) AS daynight, CAST(sensor_family AS VARCHAR) AS sensor_family,
    CAST(h3_res7 AS VARCHAR) AS h3_res7, CAST(center_lat AS DOUBLE) AS center_lat,
    CAST(center_lon AS DOUBLE) AS center_lon, CAST(country AS VARCHAR) AS country,
    CAST(type AS TINYINT) AS type,
    CAST(LEAST(GREATEST(bright_mir, 0.0), 367.0) AS FLOAT) AS bright_mir,
    CAST(GREATEST(bright_tir, 0.0) AS FLOAT) AS bright_tir,
    CAST(GREATEST(frp, 0.0) AS FLOAT) AS frp,
    CAST(point_count AS INTEGER) AS point_count,
    CAST(conf_l AS INTEGER) AS conf_l, CAST(conf_n AS INTEGER) AS conf_n,
    CAST(conf_h AS INTEGER) AS conf_h, CAST(confidence AS VARCHAR) AS confidence,
    CAST(conf_tier AS VARCHAR) AS conf_tier
"""


def get_system_ram_gb() -> float:
    if os.path.exists("/proc/meminfo"):
        with open("/proc/meminfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("MemTotal:"):
                    return int(line.split()[1]) / (1024.0 ** 2)
    return 16.0


def predict_cpu_arrays(df: pd.DataFrame, model_bundle: dict, chunk_size: int = 400_000) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = len(df)
    out_mir = np.empty(n, dtype=np.float32)
    out_tir = np.empty(n, dtype=np.float32)
    out_frp = np.empty(n, dtype=np.float32)

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        Xc = build_feature_matrix(df.iloc[start:end])
        for key, out_arr in [("bright_mir", out_mir), ("bright_tir", out_tir), ("frp_log1p", out_frp)]:
            spec = model_bundle["models"][key]
            lo, hi, w = spec["clip_lo"], spec["clip_hi"], spec["weights"]
            cols = []
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                for m_name in model_bundle["tree_names"]:
                    cols.append(np.clip(spec["trees"][m_name].predict(Xc), lo, hi))
            pred = np.clip(np.column_stack(cols) @ w, lo, hi)
            if spec["eqm_map"] is not None:
                pred = np.clip(apply_eqm_map(pred, spec["eqm_map"]), lo, hi)
            if spec["is_log"]:
                pred = np.maximum(np.expm1(pred), 0.0)
            out_arr[start:end] = pred.astype(np.float32)

    return out_mir, out_tir, out_frp


def harmonize_global_year_cpu(year: int, model_bundle: dict | None = None, duck_threads: int = N_CPU_JOBS) -> tuple[int, int, int, int, float]:
    t0 = time.time()
    active_bundle = model_bundle if model_bundle is not None else joblib.load(MODEL_OUT_PATH)
    mod_pq = f"{H3_WORLD_ROOT}/modis/{year}.parquet"
    vii_pq = f"{H3_WORLD_ROOT}/viirs/{year}.parquet"
    has_viirs = os.path.exists(vii_pq)
    out_pq = f"{HARMONIZED_WORLD_ROOT}/{year}.parquet"
    tmp_pq = f"{HARMONIZED_WORLD_ROOT}/{year}.tmp.parquet"

    con = duckdb.connect()
    try:
        con.execute(f"SET threads={duck_threads};")
        con.execute("SET memory_limit='4GB';")
        if has_viirs:
            df_mod = con.execute(f"""
                WITH v_keys AS (SELECT DISTINCT h3_res7, acq_date, daynight FROM read_parquet('{vii_pq}'))
                SELECT m.* FROM read_parquet('{mod_pq}') m
                ANTI JOIN v_keys vk ON m.h3_res7 = vk.h3_res7 AND m.acq_date = vk.acq_date AND m.daynight = vk.daynight
            """).df()
        else:
            df_mod = con.execute(f"SELECT * FROM read_parquet('{mod_pq}')").df()

        cal_cnt = len(df_mod)
        parts = []
        if has_viirs:
            parts.append(f"SELECT {SCHEMA_COLS_SQL} FROM read_parquet('{vii_pq}')")
        if cal_cnt > 0:
            pmir, ptir, pfrp = predict_cpu_arrays(df_mod, active_bundle)
            df_mod["bright_mir"], df_mod["bright_tir"], df_mod["frp"] = pmir, ptir, pfrp
            df_mod["sensor_family"] = "MODIS_CALIBRATED"
            con.register("df_mod_cal", df_mod)
            parts.append(f"SELECT {SCHEMA_COLS_SQL} FROM df_mod_cal")

        con.execute(f"COPY ({' UNION ALL '.join(parts)}) TO '{tmp_pq}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000);")
        tot, vii_cnt, bad_rows = con.execute(f"""
            SELECT
                COUNT(*),
                COALESCE(SUM(CASE WHEN sensor_family='VIIRS' THEN 1 ELSE 0 END), 0),
                COALESCE(SUM(CASE WHEN frp < 0 OR isnan(frp) OR bright_mir IS NULL OR bright_tir IS NULL OR conf_l + conf_n + conf_h != point_count THEN 1 ELSE 0 END), 0)
            FROM read_parquet('{tmp_pq}')
        """).fetchone()
        if tot <= 0 or bad_rows > 0:
            raise RuntimeError(f"Integrity check failed for {year}: tot={tot}, bad_rows={bad_rows}")
        os.replace(tmp_pq, out_pq)
        return year, int(tot), int(vii_cnt), cal_cnt, time.time() - t0
    finally:
        if os.path.exists(tmp_pq):
            os.remove(tmp_pq)
        con.close()


years = list(range(2025, 2011, -1)) + list(range(2011, 1999, -1))
ram_gb = get_system_ram_gb()
use_mp = (os.name != "nt" and ram_gb >= 24.0 and N_CPU_JOBS >= 4)

t_all = time.time()
tot_all, vii_all, cal_all = 0, 0, 0

if use_mp:
    n_workers = min(4, max(2, N_CPU_JOBS // 2))
    print(f"Exporting 26 years via ProcessPoolExecutor ({n_workers} workers, RAM={ram_gb:.1f} GB)...")
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(harmonize_global_year_cpu, yr, None, 2): yr for yr in years}
        for idx, fut in enumerate(as_completed(futures), 1):
            yr, tot, vii_cnt, cal_cnt, dt = fut.result()
            tot_all += tot; vii_all += vii_cnt; cal_all += cal_cnt
            print(f"[{idx:2d}/26] {yr}: total={tot:,} (VIIRS={vii_cnt:,}, MODIS_CALIBRATED={cal_cnt:,}) | {dt:.1f}s")
else:
    print(f"Exporting 26 years using {N_CPU_JOBS} multi-threaded CPU cores...")
    for idx, yr in enumerate(years, 1):
        _, tot, vii_cnt, cal_cnt, dt = harmonize_global_year_cpu(yr, bundle, N_CPU_JOBS)
        tot_all += tot; vii_all += vii_cnt; cal_all += cal_cnt
        print(f"[{idx:2d}/26] {yr}: total={tot:,} (VIIRS={vii_cnt:,}, MODIS_CALIBRATED={cal_cnt:,}) | {dt:.1f}s")

print(f"\nALL GLOBAL CPU DONE in {time.time() - t_all:.1f}s | total={tot_all:,} (VIIRS={vii_all:,}, MODIS_CALIBRATED={cal_all:,})")

Exporting 26 years via ProcessPoolExecutor (4 workers, RAM=51.0 GB)...
